# Feature Engineering de imágenes de palta Hass

**Autor:** Luis Daniel Paliza Vilca  
**Etapa:** preparación de características  
**Modelo principal:** ResNet50 mediante Transfer Learning, con TensorFlow/Keras

## 1. Objetivo

Preparar una representación reproducible de las imágenes para ResNet50 y examinar
descriptores de color y textura que ayuden a justificar las transformaciones.
Cada archivo conserva su etiqueta y su partición. Sus tres franjas corresponden a
vistas de una misma palta; no se convierten en muestras independientes.

| EDA | Feature Engineering |
|---|---|
| Comprende y audita el dataset: clases, dimensiones, calidad, duplicados y leakage. | Define y aplica transformaciones: tamaño, escala, color, textura y entrada del modelo. |
| Se conserva en `4. EDA/01_EDA.ipynb`. | Se desarrolla en este notebook, sin entrenar. |

Los descriptores manuales sirven para caracterizar la representación. ResNet50 recibe
imágenes y aprende sus propias características; este avance no concatena la tabla
de color y textura a la red.

**Alcance de la entrega:** el RAR incluye el EDA y sus reportes, pero no los TIFF.
Las celdas permiten leer la metodología sin el dataset y muestran qué resultados
quedan pendientes. Con `DATASET_PALTA` disponible, **Ejecutar todo** calcula las
transformaciones sobre una muestra reproducible de `train`. No hay imágenes sintéticas
ni métricas de clasificación presentadas como resultados de este proyecto.


## 2. Carga de imágenes

La búsqueda admite `DATASET_PALTA` dentro o junto al proyecto. Si existen ambas
ubicaciones, indicar una ruta relativa en la configuración. Se conservan `train`,
`validation` y `test` tal como están.

Se preparan hasta 12 archivos por clase de `train` para mantener ágil este avance.
Es una muestra equilibrada para inspeccionar transformaciones, no un cambio de
distribución del entrenamiento. El notebook no lee imágenes de validación ni test.


In [1]:
from pathlib import Path
import sys
import os
import json
import hashlib
import importlib.util
import importlib.metadata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import rgb_to_hsv
from IPython.display import display, Markdown

raiz_candidata = next((p for p in (Path.cwd(), *Path.cwd().parents)
                      if (p / "utilidades_imagenes.py").is_file()
                      and (p / "README.md").is_file()), None)
if raiz_candidata is None:
    raise FileNotFoundError("Abra el notebook dentro de la carpeta del proyecto.")
if str(raiz_candidata) not in sys.path:
    sys.path.insert(0, str(raiz_candidata))

from utilidades_imagenes import (
    encontrar_raiz, resolver_dataset, listar_imagenes, cargar_rgb,
    ajustar_tamano, crear_dataset_tf, TAMANO_ENTRADA,
)

RAIZ = encontrar_raiz()
SEMILLA = 42
RUTA_DATASET_RELATIVA = None  # Ejemplo: "../DATASET_PALTA".
MUESTRAS_POR_CLASE = 12
TAMANO = TAMANO_ENTRADA  # (alto, ancho) = (256, 512).
VALIDAR_CON_TENSORFLOW = True
EXTRAER_CARACTERISTICAS_PROFUNDAS = False
GUARDAR_RESULTADOS = True
SALIDA = RAIZ / "7. resultados" / "feature_engineering"

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.2})
DATASET = resolver_dataset(RAIZ, RUTA_DATASET_RELATIVA, requerido=False)
columnas_muestra = ["archivo", "particion", "clase", "etiqueta", "id_archivo"]
tabla_train = pd.DataFrame(columns=columnas_muestra)
tabla_muestra = tabla_train.copy()

if DATASET is None:
    print("Dataset no disponible. Las transformaciones sobre imágenes reales quedan pendientes.")
else:
    tabla_train = listar_imagenes(DATASET, "train")
    if tabla_train.empty:
        raise ValueError("No se encontraron imágenes de entrenamiento.")
    if MUESTRAS_POR_CLASE < 1:
        raise ValueError("MUESTRAS_POR_CLASE debe ser positivo.")
    grupos = [g.sample(n=min(len(g), MUESTRAS_POR_CLASE), random_state=SEMILLA)
              for _, g in tabla_train.groupby("clase", sort=True)]
    tabla_muestra = pd.concat(grupos).sort_values(["clase", "archivo"]).reset_index(drop=True)
    print("Dataset:", Path(os.path.relpath(DATASET, RAIZ)).as_posix())
    print("Archivos de train seleccionados:", len(tabla_muestra))
    display(tabla_muestra)


Dataset no disponible. Las transformaciones sobre imágenes reales quedan pendientes.


### Antecedente que condiciona la siguiente etapa

Los reportes de EDA adjuntos corresponden a la ejecución documentada del 9 de octubre
de 2026. Registran 829 imágenes y dos pares exactos entre particiones, uno con etiquetas
distintas. La tabla siguiente lee ese reporte; no vuelve a auditar el dataset.

Estas incidencias no impiden estudiar transformaciones sobre `train`, pero deben
resolverse antes de entrenar o interpretar métricas. No se corrigen moviendo archivos
en este notebook. La identidad física por fruto/lote aún requiere documentación.


In [2]:
ruta_resumen_eda = RAIZ / "7. resultados" / "eda" / "eda_resumen.json"
resumen_eda = {}
if ruta_resumen_eda.is_file():
    resumen_eda = json.loads(ruta_resumen_eda.read_text(encoding="utf-8"))
    claves = ["imagenes", "pares_exactos_entre_particiones", "pares_exactos_etiquetas_distintas"]
    display(pd.Series({k: resumen_eda.get(k) for k in claves}, name="Reporte EDA guardado"))
else:
    print("No se encontró el reporte EDA. Revise primero el notebook de exploración.")


imagenes                             829
pares_exactos_entre_particiones        2
pares_exactos_etiquetas_distintas      1
Name: Reporte EDA guardado, dtype: int64

## 3. Revisión de dimensiones para la entrada

Se mide la muestra de `train` después de orientar la imagen según EXIF y convertirla
a RGB en memoria. Este control determina cómo ajustar el tamaño; el análisis completo
de resoluciones, formatos y calidad permanece en el EDA.

`id_archivo` conserva la ruta relativa del archivo. Es un identificador de procedencia,
no una prueba de que dos nombres distintos correspondan a frutos diferentes.


In [3]:
registros_dimensiones = []
for fila in tabla_muestra.itertuples(index=False):
    imagen = cargar_rgb(DATASET / fila.archivo)
    registros_dimensiones.append({"archivo": fila.archivo, "clase": fila.clase,
                                 "ancho": imagen.width, "alto": imagen.height,
                                 "aspecto": imagen.width / imagen.height})
dimensiones = pd.DataFrame(registros_dimensiones)
if not dimensiones.empty:
    display(dimensiones.groupby("clase")[["ancho", "alto", "aspecto"]].agg(["min", "median", "max"]))
else:
    print("Dimensiones de la muestra: pendientes de cargar los TIFF originales.")


Dimensiones de la muestra: pendientes de cargar los TIFF originales.


## 4. Preparación del tamaño de entrada

Se conserva el punto de partida del baseline: **256 píxeles de alto × 512 de ancho**.
La imagen se reduce con interpolación Lanczos y se centra en un lienzo de ese tamaño.
El relleno evita deformar la palta y conserva todas las franjas; no se recorta el defecto.
El color RGB del relleno es `(124, 117, 104)`, aproximación entera de las medias de
ImageNet empleadas por el preprocesamiento, para dejar valores próximos a cero después
del centrado.

También se muestra `224 × 224` como comparación visual. No es obligatorio usar una
entrada cuadrada con `include_top=False`: ResNet50 admite tres canales y dimensiones
de al menos 32 píxeles [1]. El tamaño elegido sigue siendo provisional: hay que revisar
defectos pequeños, proporción de relleno y memoria antes de fijarlo.

La máscara marca el área ocupada por la imagen redimensionada y excluye **solo el
relleno** del cálculo de descriptores. No segmenta la fruta; los rodillos y el fondo
original siguen presentes. La CNN recibe el lienzo completo, incluido su relleno.


In [4]:
preparadas = []
for fila in tabla_muestra.itertuples(index=False):
    imagen = cargar_rgb(DATASET / fila.archivo)
    matriz, mascara = ajustar_tamano(imagen, TAMANO)
    preparadas.append({"archivo": fila.archivo, "clase": fila.clase,
                       "etiqueta": int(fila.etiqueta), "particion": "train",
                       "id_archivo": fila.id_archivo, "rgb": matriz, "mascara": mascara})

if preparadas:
    ejemplos = tabla_muestra.groupby("clase", sort=True).head(1)
    fig, ejes = plt.subplots(len(ejemplos), 3, figsize=(15, 3.5 * len(ejemplos)), squeeze=False)
    for fila, ejes_fila in zip(ejemplos.itertuples(index=False), ejes):
        original = cargar_rgb(DATASET / fila.archivo)
        cuadrada, _ = ajustar_tamano(original, (224, 224))
        rectangular, mascara = ajustar_tamano(original, TAMANO)
        titulos = [f"{fila.clase}: original {original.height} × {original.width}",
                   "224 × 224 con relleno", f"256 × 512; relleno {1 - mascara.mean():.1%}"]
        for eje, imagen, titulo in zip(ejes_fila, [original, cuadrada, rectangular], titulos):
            eje.imshow(imagen)
            eje.set_title(titulo, fontsize=10)
            eje.axis("off")
    fig.suptitle("Comparación de entrada: alto × ancho")
    plt.tight_layout()
    plt.show()


## 5. Normalización

Hay dos representaciones con usos distintos:

| Representación | Operación | Uso |
|---|---|---|
| RGB en `[0, 1]` | `rgb.astype(float32) / 255` | Histogramas y descriptores de este avance. |
| Entrada de ResNet50 v1 | RGB `[0, 255]` → BGR → resta de medias ImageNet | Extractor preentrenado de Keras. |

Para ResNet50 no se divide entre 255 antes de `preprocess_input`, ni se aplica otra
normalización después [1, 2]. Las medias de ImageNet son constantes del modelo;
no se estiman usando validation o test. Si más adelante se entrena un clasificador
con la tabla de descriptores, su escalador deberá ajustarse exclusivamente con train.


In [5]:
def normalizar_rgb(matriz):
    return matriz.astype(np.float32) / np.float32(255.0)


def preparar_resnet50_numpy(matriz):
    # Equivalente numérico al preprocesamiento de ResNet50 v1.
    bgr = matriz.astype(np.float32)[..., ::-1].copy()
    return bgr - np.array([103.939, 116.779, 123.680], dtype=np.float32)


if preparadas:
    rgb = preparadas[0]["rgb"]
    rgb_01 = normalizar_rgb(rgb)
    entrada_resnet = preparar_resnet50_numpy(rgb)
    display(pd.DataFrame([
        {"representacion": "RGB original", "tipo": str(rgb.dtype), "min": rgb.min(), "max": rgb.max()},
        {"representacion": "RGB descriptivo", "tipo": str(rgb_01.dtype), "min": rgb_01.min(), "max": rgb_01.max()},
        {"representacion": "Entrada ResNet50", "tipo": str(entrada_resnet.dtype),
         "min": entrada_resnet.min(), "max": entrada_resnet.max()},
    ]))


## 6. Canales RGB

Se separan los canales para observar qué información cambia entre ellos. La misma
muestra conserva sus tres canales al entrar al modelo; no se entrena una red por canal.
Las estadísticas se calculan sobre los píxeles señalados por la máscara.


In [6]:
if preparadas:
    ejemplo = preparadas[0]
    rgb_01 = normalizar_rgb(ejemplo["rgb"])
    fig, ejes = plt.subplots(1, 4, figsize=(15, 4))
    ejes[0].imshow(ejemplo["rgb"])
    ejes[0].set_title(f"{ejemplo['clase']} · RGB")
    for canal, (nombre, color) in enumerate(zip(["Rojo", "Verde", "Azul"], ["Reds", "Greens", "Blues"])):
        ejes[canal + 1].imshow(rgb_01[..., canal], cmap=color, vmin=0, vmax=1)
        ejes[canal + 1].set_title(nombre)
    for eje in ejes:
        eje.axis("off")
    plt.tight_layout()
    plt.show()


## 7. Brillo

Se utiliza la aproximación `Y = 0.299R + 0.587G + 0.114B` sobre RGB normalizado.
Su promedio describe intensidad de la imagen; no mide iluminación física ni calidad
del defecto. Conserva influencia del fondo original.


In [7]:
def calcular_gris(rgb_01):
    return rgb_01 @ np.array([0.299, 0.587, 0.114], dtype=np.float32)


def medir_brillo(matriz, mascara):
    valores = calcular_gris(normalizar_rgb(matriz))[mascara]
    return {"brillo_media": float(valores.mean()), "brillo_mediana": float(np.median(valores))}


tabla_brillo = pd.DataFrame([{"archivo": e["archivo"], "clase": e["clase"],
                             **medir_brillo(e["rgb"], e["mascara"])} for e in preparadas])
if not tabla_brillo.empty:
    display(tabla_brillo.groupby("clase")[["brillo_media", "brillo_mediana"]].mean().round(4))


## 8. Contraste

Se usan la desviación estándar de `Y` (contraste RMS) y la diferencia entre sus
percentiles 90 y 10. Se calculan como descriptores; no se aplica ecualización,
CLAHE o corrección gamma automática. Modificar contraste o color podría alterar
indicios de russet o picadura, y requiere comparar ejemplos reales.


In [8]:
def medir_contraste(matriz, mascara):
    valores = calcular_gris(normalizar_rgb(matriz))[mascara]
    p10, p90 = np.percentile(valores, [10, 90])
    return {"contraste_rms": float(valores.std()), "contraste_p90_p10": float(p90 - p10)}


tabla_contraste = pd.DataFrame([{"archivo": e["archivo"], "clase": e["clase"],
                                **medir_contraste(e["rgb"], e["mascara"])} for e in preparadas])
if not tabla_contraste.empty:
    display(tabla_contraste.groupby("clase")[["contraste_rms", "contraste_p90_p10"]].mean().round(4))


## 9. Histogramas básicos

Cada canal RGB se resume con 16 intervalos entre 0 y 1. Los conteos se dividen entre
el número de píxeles válidos, de modo que cada histograma suma 1. Esta normalización
permite comparar imágenes de distintos tamaños de contenido. Cada imagen tendrá
48 características de histograma, además de sus estadísticas de color.


In [9]:
def extraer_histogramas(matriz, mascara, intervalos=16):
    pixeles = normalizar_rgb(matriz)[mascara]
    rasgos = {}
    for canal, nombre in enumerate(("r", "g", "b")):
        conteos, _ = np.histogram(pixeles[:, canal], bins=intervalos, range=(0, 1))
        frecuencias = conteos / conteos.sum()
        rasgos.update({f"hist_{nombre}_{i:02d}": float(v) for i, v in enumerate(frecuencias)})
    return rasgos


if preparadas:
    ejemplos = [next(e for e in preparadas if e["clase"] == clase)
                for clase in sorted({e["clase"] for e in preparadas})]
    fig, ejes = plt.subplots(1, len(ejemplos), figsize=(6 * len(ejemplos), 4), squeeze=False)
    centros = (np.arange(16) + 0.5) / 16
    for eje, ejemplo in zip(ejes[0], ejemplos):
        histograma = extraer_histogramas(ejemplo["rgb"], ejemplo["mascara"])
        for canal, color in zip(("r", "g", "b"), ("red", "green", "blue")):
            eje.plot(centros, [histograma[f"hist_{canal}_{i:02d}"] for i in range(16)], color=color, label=canal.upper())
        eje.set(title=ejemplo["clase"], xlabel="Intensidad RGB normalizada", ylabel="Frecuencia relativa")
        eje.legend()
    plt.tight_layout()
    plt.show()


## 10. Características de color

Se extraen media y desviación estándar de RGB y de saturación/valor en HSV.
El matiz es circular: 0 y 1 representan colores vecinos. Por ello se resume mediante
seno y coseno ponderados por saturación, en lugar de promediar H directamente.
Los píxeles prácticamente grises (`S < 0.05`) no aportan al resumen de matiz.

HSV se usa para obtener descriptores interpretables; la entrada de ResNet50 continúa
siendo RGB. No hay selección de variables ni evaluación de capacidad predictiva en
este avance.


In [10]:
def extraer_color(matriz, mascara):
    rgb_01 = normalizar_rgb(matriz)
    pixeles = rgb_01[mascara]
    hsv = rgb_to_hsv(rgb_01)[mascara]
    rasgos = {}
    for indice, canal in enumerate(("r", "g", "b")):
        rasgos[f"{canal}_media"] = float(pixeles[:, indice].mean())
        rasgos[f"{canal}_std"] = float(pixeles[:, indice].std())
    for indice, canal in ((1, "s"), (2, "v")):
        rasgos[f"{canal}_media"] = float(hsv[:, indice].mean())
        rasgos[f"{canal}_std"] = float(hsv[:, indice].std())
    pesos = np.where(hsv[:, 1] >= 0.05, hsv[:, 1], 0.0)
    angulos = 2 * np.pi * hsv[:, 0]
    suma_pesos = pesos.sum()
    rasgos["h_seno"] = float(np.sum(pesos * np.sin(angulos)) / suma_pesos) if suma_pesos else 0.0
    rasgos["h_coseno"] = float(np.sum(pesos * np.cos(angulos)) / suma_pesos) if suma_pesos else 0.0
    rasgos["fraccion_con_matiz"] = float(np.mean(hsv[:, 1] >= 0.05))
    return rasgos


if preparadas:
    display(pd.Series(extraer_color(preparadas[0]["rgb"], preparadas[0]["mascara"]), name="Ejemplo de color"))


## 11. Posibles características de textura

Se implementa una matriz de coocurrencia de niveles de gris (GLCM) sencilla: 16 niveles,
distancia de un píxel, vecinos horizontales y verticales, conteo simétrico y normalizado.
Solo se cuentan pares cuyos dos píxeles pertenecen al área de imagen. Se obtienen
contraste, homogeneidad, segundo momento angular (ASM) y entropía [3].

Estas variables describen relaciones locales de intensidad. Dependen de la resolución,
la interpolación, los rodillos y el fondo; no demuestran por sí solas presencia de
russet o picadura. La implementación con NumPy evita incorporar otra dependencia
para este avance. LBP o descriptores de una región de fruta pueden evaluarse después,
manteniendo su identidad y sin tratarlos como observaciones independientes.


In [11]:
def extraer_textura(matriz, mascara, niveles=16):
    gris = calcular_gris(normalizar_rgb(matriz))
    cuantizada = np.clip((gris * niveles).astype(np.int32), 0, niveles - 1)
    glcm = np.zeros((niveles, niveles), dtype=np.float64)
    vecinos = [
        (cuantizada[:, :-1], cuantizada[:, 1:], mascara[:, :-1] & mascara[:, 1:]),
        (cuantizada[:-1, :], cuantizada[1:, :], mascara[:-1, :] & mascara[1:, :]),
    ]
    for origen, destino, validos in vecinos:
        indices = origen[validos] * niveles + destino[validos]
        conteos = np.bincount(indices, minlength=niveles * niveles).reshape(niveles, niveles)
        glcm += conteos + conteos.T
    if glcm.sum() == 0:
        raise ValueError("No hay pares válidos para medir textura.")
    glcm /= glcm.sum()
    i, j = np.indices(glcm.shape)
    probabilidades = glcm[glcm > 0]
    return {
        "glcm_contraste": float(np.sum(glcm * (i - j) ** 2)),
        "glcm_homogeneidad": float(np.sum(glcm / (1 + (i - j) ** 2))),
        "glcm_asm": float(np.sum(glcm ** 2)),
        "glcm_entropia": float(-np.sum(probabilidades * np.log2(probabilidades))),
    }


registros_caracteristicas = []
for ejemplo in preparadas:
    matriz, mascara = ejemplo["rgb"], ejemplo["mascara"]
    datos = {k: ejemplo[k] for k in ("archivo", "particion", "clase", "etiqueta", "id_archivo")}
    datos.update({"alto_entrada": TAMANO[0], "ancho_entrada": TAMANO[1],
                  "fraccion_relleno": float(1 - mascara.mean())})
    for extraer in (medir_brillo, medir_contraste, extraer_histogramas, extraer_color, extraer_textura):
        datos.update(extraer(matriz, mascara))
    registros_caracteristicas.append(datos)
caracteristicas = pd.DataFrame(registros_caracteristicas)
if not caracteristicas.empty:
    display(caracteristicas.head())
    columnas_resumen = ["brillo_media", "contraste_rms", "s_media", "glcm_contraste", "glcm_homogeneidad"]
    display(caracteristicas.groupby("clase")[columnas_resumen].mean().round(4))
    valores = caracteristicas.select_dtypes(include="number").to_numpy()
    assert np.isfinite(valores).all(), "Revise descriptores no finitos."


## 12. Data augmentation planteada

La transformación inicial es un volteo horizontal aleatorio con probabilidad 0.5.
Conserva las tres franjas en su orden vertical y no recorta la imagen. Su invariancia
de etiqueta debe confirmarse con el especialista: las diferencias por cámara o por
sentido de giro podrían ser relevantes.

| Transformación | Decisión para este avance |
|---|---|
| Volteo horizontal | Propuesta inicial para train; vista previa en memoria. |
| Volteo vertical / rotación grande | No se propone porque cambia el orden u orientación de las franjas. |
| Recortes y zoom | Pendientes de verificar que el defecto permanezca visible. |
| Brillo, contraste, saturación | Pendientes de revisión; no se activan automáticamente. |
| Validation / test | Resize y preprocesamiento deterministas; sin augmentation. |

No se guardan copias aumentadas ni se cambia la cantidad de frutos independientes.
No se aplica oversampling o pesos de clase en este notebook; el desbalance se atenderá
en la etapa de entrenamiento después de revisar el EDA.


In [12]:
def aumentar_horizontal(matriz, generador):
    voltear = bool(generador.random() < 0.5)
    resultado = np.flip(matriz, axis=1).copy() if voltear else matriz.copy()
    return resultado, voltear


if preparadas:
    generador = np.random.default_rng(SEMILLA)
    ejemplo = preparadas[0]
    fig, ejes = plt.subplots(1, 4, figsize=(15, 4))
    ejes[0].imshow(ejemplo["rgb"])
    ejes[0].set_title("Original preparada")
    for eje in ejes[1:]:
        aumentada, volteada = aumentar_horizontal(ejemplo["rgb"], generador)
        eje.imshow(aumentada)
        eje.set_title("Volteo horizontal" if volteada else "Sin volteo")
    for eje in ejes:
        eje.axis("off")
    plt.tight_layout()
    plt.show()


## 13. Preparación para ResNet50

El contrato de entrada será `float32`, forma `(lote, 256, 512, 3)`, RGB en `[0, 255]`.
La carga de TIFF y el resize están compartidos con `6. entrenamiento/resnet50.ipynb`
mediante `utilidades_imagenes.py`. El pipeline carga por lotes y no modifica los archivos.

El modelo futuro aplicará el volteo únicamente durante entrenamiento y luego
`tensorflow.keras.applications.resnet50.preprocess_input` **una sola vez**.
La celda siguiente verifica la equivalencia con el cálculo NumPy y prueba un lote
de train si TensorFlow está instalado. No construye ni entrena una red.

Instalación opcional, desde la raíz: `python -m pip install -r requirements_resnet50.txt`.
Si no está instalado, los descriptores, histogramas y la demostración numérica siguen
funcionando con las dependencias básicas.


In [13]:
tensorflow_disponible = importlib.util.find_spec("tensorflow") is not None
validacion_tensorflow = "pendiente"
if not preparadas:
    print("Prueba del lote: pendiente de cargar imágenes de train.")
elif not VALIDAR_CON_TENSORFLOW:
    print("Prueba con TensorFlow desactivada en la configuración.")
elif not tensorflow_disponible:
    print("TensorFlow no está instalado. Se conserva la preparación NumPy; validación del lote pendiente.")
else:
    import tensorflow as tf
    from tensorflow.keras.applications.resnet50 import preprocess_input

    tf.keras.utils.set_random_seed(SEMILLA)
    matriz_original = preparadas[0]["rgb"].copy()
    referencia = preprocess_input(matriz_original.astype(np.float32).copy(), data_format="channels_last")
    np.testing.assert_allclose(preparar_resnet50_numpy(matriz_original), referencia, atol=1e-5)
    np.testing.assert_array_equal(matriz_original, preparadas[0]["rgb"])

    flujo = crear_dataset_tf(tabla_muestra, DATASET, tamano=TAMANO,
                             tamano_lote=4, entrenamiento=False, semilla=SEMILLA)
    lote_rgb, etiquetas = next(iter(flujo.take(1)))
    lote_preparado = preprocess_input(lote_rgb)
    assert tuple(lote_rgb.shape[1:]) == (*TAMANO, 3)
    assert lote_rgb.dtype == tf.float32
    assert bool(tf.reduce_all(tf.math.is_finite(lote_preparado)))
    np.testing.assert_array_equal(lote_rgb[0].numpy(), preparadas[0]["rgb"].astype(np.float32))
    validacion_tensorflow = "correcta"
    print("Lote RGB:", tuple(lote_rgb.shape), "| Etiquetas:", etiquetas.numpy())
    print("Preprocesamiento ResNet50 verificado; no se entrenó ningún modelo.")


Prueba del lote: pendiente de cargar imágenes de train.


## 14. Posible extracción futura de características profundas

Una alternativa posterior es usar `ResNet50(weights="imagenet", include_top=False,
pooling="avg")` como extractor congelado. El promedio espacial entrega un vector de
**2048 valores por archivo**, manteniendo una fila por palta [1]. Esos valores son
características aprendidas en ImageNet, no descriptores manuales de defecto.

La función queda disponible, pero su llamada está desactivada. Activarla descargará
los pesos de ImageNet la primera vez, si no están en caché, y hará inferencia sobre
la muestra de train; no entrenará. No se sustituye `weights="imagenet"` por pesos
aleatorios para presentar vectores como características transferidas.

Si se trabaja más adelante por cámara o por vista, todas las representaciones de
un mismo fruto conservarán el mismo grupo y partición. La agregación y la evaluación
deben realizarse por fruto, no contando cada franja como una muestra independiente.


In [14]:
def extraer_caracteristicas_profundas(tabla, dataset, tamano=TAMANO, tamano_lote=4):
    import tensorflow as tf
    from tensorflow.keras.applications.resnet50 import preprocess_input

    extractor = tf.keras.applications.ResNet50(
        weights="imagenet", include_top=False, pooling="avg", input_shape=(*tamano, 3)
    )
    extractor.trainable = False
    flujo = crear_dataset_tf(tabla, dataset, tamano, tamano_lote, entrenamiento=False)
    bloques = []
    for lote, _ in flujo:
        vectores = extractor(preprocess_input(lote), training=False).numpy()
        bloques.append(vectores)
    matriz = np.concatenate(bloques, axis=0)
    assert matriz.shape == (len(tabla), 2048)
    columnas = [f"resnet50_{i:04d}" for i in range(matriz.shape[1])]
    identidad = tabla[["archivo", "particion", "clase", "etiqueta", "id_archivo"]].reset_index(drop=True)
    return pd.concat([identidad, pd.DataFrame(matriz, columns=columnas)], axis=1)


caracteristicas_profundas = None
if EXTRAER_CARACTERISTICAS_PROFUNDAS:
    if not preparadas:
        raise ValueError("Se necesitan imágenes de train para extraer características profundas.")
    if not tensorflow_disponible:
        raise ImportError("Instale requirements_resnet50.txt para usar el extractor.")
    caracteristicas_profundas = extraer_caracteristicas_profundas(tabla_muestra, DATASET)
    display(caracteristicas_profundas.iloc[:3, :9])
else:
    print("Extracción profunda desactivada. No se descargan pesos ni se construye ResNet50.")


Extracción profunda desactivada. No se descargan pesos ni se construye ResNet50.


### Registro del avance

Cuando hay imágenes, se exportan una tabla de descriptores y un resumen de configuración
a `7. resultados/feature_engineering/`. Se conservan rutas relativas, clases y hashes
de la muestra para identificar la entrada. Los reportes se regeneran al ejecutar;
las imágenes originales permanecen intactas. La tabla profunda solo se guarda si
se solicitó explícitamente su extracción.


In [15]:
def sha256_archivo(ruta):
    calculo = hashlib.sha256()
    with Path(ruta).open("rb") as archivo:
        for bloque in iter(lambda: archivo.read(1024 * 1024), b""):
            calculo.update(bloque)
    return calculo.hexdigest()


if GUARDAR_RESULTADOS and not caracteristicas.empty:
    SALIDA.mkdir(parents=True, exist_ok=True)
    caracteristicas.to_csv(SALIDA / "fe_caracteristicas_train.csv", index=False, encoding="utf-8-sig")
    fuentes = [{"archivo": a, "sha256": sha256_archivo(DATASET / a)} for a in tabla_muestra["archivo"]]
    configuracion = {
        "etapa": "feature_engineering", "modelo": "ResNet50", "framework": "TensorFlow/Keras",
        "dataset_relativo": Path(os.path.relpath(DATASET, RAIZ)).as_posix(),
        "particion_utilizada": "train", "cantidad_muestra": len(tabla_muestra),
        "muestras_por_clase_maximo": MUESTRAS_POR_CLASE, "semilla": SEMILLA,
        "tamano_alto_ancho": list(TAMANO), "resize": "Lanczos con aspecto conservado y relleno",
        "relleno_rgb": [124, 117, 104], "mascara": "excluye relleno; incluye fondo original",
        "normalizacion_descriptores": "RGB / 255", "entrada_modelo": "RGB float32 0-255",
        "preprocesamiento_modelo": "ResNet50 v1: BGR menos [103.939, 116.779, 123.680]",
        "augmentation_propuesta": "volteo horizontal p=0.5, solo train",
        "glcm": {"niveles": 16, "distancia": 1, "direcciones": ["horizontal", "vertical"], "simetrica": True},
        "validacion_tensorflow": validacion_tensorflow,
        "extraccion_profunda": caracteristicas_profundas is not None,
        "entrenamiento_ejecutado": False, "archivos_muestra": fuentes,
        "versiones": {n: importlib.metadata.version(n) for n in ["numpy", "pandas", "matplotlib", "Pillow"]},
    }
    if tensorflow_disponible and "tf" in globals():
        configuracion["versiones"]["tensorflow"] = tf.__version__
    (SALIDA / "fe_resumen.json").write_text(json.dumps(configuracion, ensure_ascii=False, indent=2), encoding="utf-8")
    if caracteristicas_profundas is not None:
        caracteristicas_profundas.to_csv(SALIDA / "fe_resnet50_train.csv", index=False, encoding="utf-8-sig")
    print("Reportes guardados en 7. resultados/feature_engineering/.")
elif not preparadas:
    print("No se generan resultados FE sin imágenes originales.")
else:
    print("GUARDAR_RESULTADOS está desactivado; resultados disponibles en memoria.")


No se generan resultados FE sin imágenes originales.


## 15. Conclusiones

- Se define una entrada RGB común para FE y el futuro baseline de ResNet50. El resize
  mantiene las tres franjas y la relación de aspecto; su resolución debe validarse
  visualmente con defectos pequeños antes del entrenamiento.
- RGB normalizado, brillo, contraste, histogramas, HSV y GLCM forman una tabla de
  características descriptivas. No se ha demostrado que separen las clases ni se
  presentan como sustituto de las características aprendidas por ResNet50.
- La máscara evita medir el relleno, pero los descriptores todavía incluyen el fondo
  industrial. Una futura región de fruta debe justificarse y evaluarse por separado.
- Se propone augmentation horizontal moderada, solo para train. No se modifican
  archivos ni particiones y cada archivo sigue siendo una observación.
- El EDA guardado reporta contaminación entre particiones. Antes de entrenar deben
  revisarse las dos coincidencias, la etiqueta contradictoria y la identidad por fruto/lote.
  Los hashes no bastan para demostrar independencia física.
- La siguiente etapa será validar el tamaño de entrada y el protocolo de evaluación,
  y después entrenar ResNet50 por Transfer Learning. F1 por clase, macro-F1, balanced
  accuracy, precision/recall y matriz de confusión quedan para esa etapa; test no se
  usará para ajustar transformaciones o hiperparámetros.

### Estado de esta ejecución


In [16]:
if preparadas:
    texto = (f"Se prepararon **{len(preparadas)} imágenes de train** en memoria. "
             f"La tabla contiene {len(caracteristicas.columns)} columnas, incluidas identidad y configuración. "
             f"Validación TensorFlow: **{validacion_tensorflow}**. No se entrenó ningún clasificador.")
else:
    texto = ("**Pendiente de ejecución sobre los TIFF originales.** La metodología y el código están preparados; "
             "esta ejecución no produce descriptores reales, embeddings ni métricas de clasificación.")
display(Markdown(texto))


**Pendiente de ejecución sobre los TIFF originales.** La metodología y el código están preparados; esta ejecución no produce descriptores reales, embeddings ni métricas de clasificación.

### Referencias técnicas

1. Keras. [ResNet models](https://keras.io/api/applications/resnet/resnet_models/).
   Arquitectura, `include_top=False`, pooling y contrato de preprocesamiento.
2. TensorFlow. [resnet.preprocess_input](https://www.tensorflow.org/api_docs/python/tf/keras/applications/resnet/preprocess_input).
   Entrada RGB 0–255 y conversión a BGR centrada por canal.
3. scikit-image. [GLCM texture features](https://scikit-image.org/docs/stable/auto_examples/features_detection/plot_glcm.html).
   Concepto de matriz de coocurrencia y descriptores de textura. La implementación
   de este notebook usa NumPy y no requiere instalar scikit-image.
